# FPL GW Points Predictor — XGBoost Baseline Model

**Goal**: Train an XGBoost model on GW 1–37 of the 2024‑25 season, then test how well it predicts GW 38 points.

**The idea in plain English**:
- Every row in our dataset = one player × one gameweek.
- Features = things we *know before* a gameweek kicks off (recent form, rolling averages, price, fixture, position).
- Target = `total_points` that player actually scored that gameweek.
- We train on 37 gameweeks, then ask "given everything we know before GW 38, how many points will each player score?"

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-darkgrid')
print("✓ Imports ready")

## 2. Load Processed Data

We load the CSV that was saved at the end of the EDA notebook (already cleaned + feature-engineered). If it doesn't exist yet, we run the full pipeline inline.

In [ ]:
import os, sys

# Allow imports from analysis/ when notebook is run as a standalone
if os.path.dirname(os.path.abspath('')) not in sys.path:
    sys.path.insert(0, os.path.dirname(os.path.abspath('')))

DATA_PATH = '../data/processed_fpl_data.csv'

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print(f"Loaded cached data: {df.shape}")
else:
    # Fall back to running the pipeline
    from fpl_pipeline import FPLPipeline
    pipeline = FPLPipeline(base_path='../data')
    pipeline.run_full_pipeline()
    df = pipeline.df_features.copy()
    print(f"Pipeline produced: {df.shape}")

print("\nSeasons:", sorted(df['season'].unique()))
print("GWs in 2024-25:", sorted(df.loc[df['season'] == '2024-25', 'GW'].unique()))
df.head(3)

## 3. Feature Selection

Only lag/rolling features are used — every one was created with `.shift(1)` so there is zero data leakage into the target `total_points`.

In [ ]:
TARGET = 'total_points'

CANDIDATE_FEATURES = [
    # Rolling averages (lag-safe)
    'last_3_avg_points', 'last_5_avg_points', 'last_10_avg_points',
    'season_avg_points', 'form_vs_average', 'points_std_last_5',
    # Minutes / availability proxy
    'avg_minutes_last_3', 'avg_minutes_last_5',
    # Home / away splits
    'home_avg_points', 'away_avg_points', 'home_away_diff',
    # Fixture difficulty
    'opponent_avg_points_conceded',
    # Static / slow-changing
    'position_encoded', 'value', 'was_home_numeric',
    # Cumulative
    'games_played',
]

# Keep only columns that actually exist in this version of the data
FEATURES = [c for c in CANDIDATE_FEATURES if c in df.columns]
missing = set(CANDIDATE_FEATURES) - set(FEATURES)
if missing:
    print(f"⚠ Missing features (will be skipped): {missing}")

print(f"Using {len(FEATURES)} features:")
print(FEATURES)

## 4. Train / Test Split

- **Train**: 2024-25 season, GW 1–37 (temporal split — no shuffle)
- **Test**: 2024-25 season, GW 38 (the final gameweek we are predicting)

In [ ]:
season_df = df[df['season'] == '2024-25'].copy()

train_df = season_df[season_df['GW'] <= 37].dropna(subset=FEATURES + [TARGET])
test_df  = season_df[season_df['GW'] == 38].dropna(subset=FEATURES + [TARGET])

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]

print(f"Train rows : {len(train_df):,}  ({train_df['GW'].nunique()} GWs)")
print(f"Test rows  : {len(test_df):,}   (GW 38)")
print(f"\nTarget distribution — Train")
print(y_train.describe().round(2))
print(f"\nTarget distribution — Test")
print(y_test.describe().round(2))

## 5. Train XGBoost Model

Gradient-boosted trees are a natural fit here: they handle mixed feature types, require no feature scaling, and are interpretable via feature importance.

In [ ]:
model = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 3,
    reg_alpha         = 0.1,   # L1 regularisation
    reg_lambda        = 1.0,   # L2 regularisation
    random_state      = 42,
    eval_metric       = 'mae',
    early_stopping_rounds = 30,
    verbosity         = 0,
)

model.fit(
    X_train, y_train,
    eval_set   = [(X_train, y_train), (X_test, y_test)],
    verbose    = 50,
)

print(f"\nBest iteration: {model.best_iteration}")

## 6. Evaluation Metrics

We evaluate on the **held-out GW 38** rows. Because the target is highly skewed (median ≈ 0), MAE is more interpretable than RMSE here.

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 40)
print("      GW 38 Prediction Results")
print("=" * 40)
print(f"  MAE  : {mae:.3f}  pts")
print(f"  RMSE : {rmse:.3f}  pts")
print(f"  R²   : {r2:.3f}")
print("=" * 40)

# Baseline: always predict the mean of the training target
naive_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
naive_mae  = mean_absolute_error(y_test, naive_pred)
print(f"\n  Naive baseline MAE (mean): {naive_mae:.3f}  pts")
print(f"  Improvement over baseline: {(naive_mae - mae):.3f}  pts")

## 7. Feature Importance

Which features did the model rely on most when making GW 38 predictions?

In [ ]:
importance_df = (
    pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_})
    .sort_values('importance', ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.RdYlGn(
    np.linspace(0.15, 0.85, len(importance_df))
)
ax.barh(importance_df['feature'], importance_df['importance'], color=colors)
ax.set_xlabel('F-score (gain)', fontsize=12)
ax.set_title('XGBoost Feature Importance — GW 38 Prediction', fontsize=14, fontweight='bold')
ax.axvline(importance_df['importance'].mean(), color='navy', linestyle='--', alpha=0.6, label='mean importance')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/feature_importance.png")

## 8. GW 38 — Predicted vs Actual Points

The diagonal dashed line is perfect prediction. Points above it were underestimated; points below were overestimated.

In [ ]:
results_df = test_df[['name', 'position', 'team']].copy().reset_index(drop=True)
results_df['actual']    = y_test.values
results_df['predicted'] = y_pred
results_df['error']     = results_df['predicted'] - results_df['actual']

# ── scatter plot ──────────────────────────────────────
pos_colors = {'GK': '#e74c3c', 'DEF': '#3498db', 'MID': '#2ecc71', 'FWD': '#f39c12'}
# Fallback if position column has different name
pos_col = 'position' if 'position' in results_df.columns else None

fig, ax = plt.subplots(figsize=(8, 8))

if pos_col:
    for pos, grp in results_df.groupby(pos_col):
        ax.scatter(grp['actual'], grp['predicted'],
                   alpha=0.45, s=25,
                   color=pos_colors.get(pos, 'grey'), label=pos)
    ax.legend(title='Position', fontsize=9)
else:
    ax.scatter(results_df['actual'], results_df['predicted'], alpha=0.4, s=25)

lim = max(results_df['actual'].max(), results_df['predicted'].max()) + 1
ax.plot([0, lim], [0, lim], 'k--', linewidth=1.2, label='perfect prediction')

# Annotate the 8 biggest absolute errors
top_errors = results_df.reindex(results_df['error'].abs().nlargest(8).index)
for _, row in top_errors.iterrows():
    ax.annotate(row['name'].split('_')[0],
                (row['actual'], row['predicted']),
                fontsize=7, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Actual Points (GW 38)', fontsize=12)
ax.set_ylabel('Predicted Points (GW 38)', fontsize=12)
ax.set_title(f'XGBoost GW 38 Predictions  |  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/gw38_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/gw38_predicted_vs_actual.png")

## 9. Top 20 Predicted Scorers for GW 38

The model's "best picks" — ranked by predicted points, with actual points alongside for context.

In [ ]:
top20 = results_df.sort_values('predicted', ascending=False).head(20).reset_index(drop=True)
top20.index += 1  # 1-based ranking
display(top20[['name', 'position', 'team', 'predicted', 'actual', 'error']].round(2))

# ── side-by-side bar chart ────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(top20))
width = 0.4

bars_pred   = ax.bar(x - width/2, top20['predicted'], width, label='Predicted', color='steelblue', alpha=0.85)
bars_actual = ax.bar(x + width/2, top20['actual'],    width, label='Actual',    color='tomato',    alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(
    [n.split('_')[0] for n in top20['name']],
    rotation=45, ha='right', fontsize=9
)
ax.set_ylabel('Points', fontsize=12)
ax.set_title('Top 20 Predicted Scorers — GW 38 (Predicted vs Actual)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../reports/top20_gw38.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/top20_gw38.png")

## 10. Training / Validation Loss Curve

Plotting MAE per iteration helps confirm the model converged and that early stopping fired at the right point.

In [ ]:
evals = model.evals_result()
train_mae = evals['validation_0']['mae']
val_mae   = evals['validation_1']['mae']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_mae, label='Train MAE',      color='steelblue', linewidth=1.5)
ax.plot(val_mae,   label='Validation MAE', color='tomato',    linewidth=1.5)
ax.axvline(model.best_iteration, color='green', linestyle='--',
           linewidth=1.2, label=f'Best iteration ({model.best_iteration})')
ax.set_xlabel('Boosting Round', fontsize=12)
ax.set_ylabel('MAE (pts)', fontsize=12)
ax.set_title('XGBoost Learning Curve — GW 38 Prediction', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../reports/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/learning_curve.png")

## Summary

| Item | Detail |
|------|--------|
| **Model** | XGBoost Regressor (gradient-boosted trees) |
| **Training data** | 2024-25 season — GW 1 to 37 |
| **Test data** | 2024-25 season — GW 38 only (temporal holdout) |
| **Target** | `total_points` per player per gameweek |
| **Features used** | Rolling averages (3/5/10 GW), form, home/away splits, fixture difficulty, position, value |
| **Data leakage** | None — all rolling features use `.shift(1)` |
| **Charts saved** | `reports/feature_importance.png`, `reports/gw38_predicted_vs_actual.png`, `reports/top20_gw38.png`, `reports/learning_curve.png` |

### Next Steps
- Cross-validate across multiple final gameweeks (GW 35–38) to get a stable MAE estimate
- Add raw FPL stats as features (e.g. `ict_index_last_3`, `xG_last_5`) once they are lag-shifted
- Cluster players by form profile and use cluster ID as an additional categorical feature
- Export the trained model with `model.save_model('../models/xgb_gw38.json')` for deployment